In [15]:
# 1. Clone the repository
!git clone --depth 1 https://github.com/EconShark/23_big_data_analysis_class.git

# 2. Copy the datasets to /content/
!cp -rn 23_big_data_analysis_class/Lab04_Clearning_Data_with_PySpark/Datasets/. /content/Datasets

# 3. Remove the cloned repository
!rm -rf 23_big_data_analysis_class

Cloning into '23_big_data_analysis_class'...
remote: Enumerating objects: 101, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 101 (delta 9), reused 94 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (101/101), 70.70 MiB | 21.46 MiB/s, done.
Resolving deltas: 100% (9/9), done.
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead


# THỰC HÀNH LÀM SẠCH DỮ LIỆU VỚI PYSPARK

## Nội dung thực hành

1. Giải thích vai trò của data cleaning trong quy trình Big Data.
2. Định nghĩa schema rõ ràng khi đọc dữ liệu.
3. Hiểu tính immutable và lazy processing của Spark.
4. Đọc/ghi dữ liệu ở định dạng CSV và Parquet.
5. Thực hiện các phép biến đổi cột của DataFrame.
6. Lọc dữ liệu lỗi, dữ liệu null và dữ liệu bất thường.
7. Dùng `when()`, `otherwise()` và UDF.
8. Hiểu partition, ID phân tán và caching.
9. Quan sát execution plan, shuffle và broadcast join.
10. Xây dựng pipeline làm sạch dữ liệu hoàn chỉnh.

<a id="setup"></a>
## 1. Chuẩn bị môi trường

### 1.1. Cài đặt PySpark

In [16]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    print("Chưa có PySpark. Đang cài đặt...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
else:
    print("PySpark đã sẵn sàng.")

PySpark đã sẵn sàng.


### 1.2. Xác định thư mục dataset

In [17]:
from pathlib import Path

candidates = [
    Path.cwd() / "Datasets",
    Path("/content/Datasets"),
    Path.cwd().parent / "Datasets"
]

DATA_DIR = next((p for p in candidates if p.exists()), None)

if DATA_DIR is None:
    print("Không tìm thấy thư mục Datasets.")
else:
    print("DATA_DIR =", DATA_DIR.resolve())
    for p in sorted(DATA_DIR.iterdir()):
        print(" -", p.name)

DATA_DIR = /content/Datasets
 - AA_DFW_2014_Departures_Short.csv.gz
 - AA_DFW_2015_Departures_Short.csv.gz
 - AA_DFW_2016_Departures_Short.csv.gz
 - AA_DFW_2017_Departures_Short.csv.gz
 - DallasCouncilVoters.csv.gz
 - DallasCouncilVotes.csv.gz


### 1.3. Khởi tạo SparkSession

In [18]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("CleaningDataWithPySpark")
    .master("local[*]")
    .getOrCreate()
)

sc = spark.sparkContext

print("Spark version :", spark.version)
print("Master        :", sc.master)
print("Parallelism   :", sc.defaultParallelism)

Spark version : 4.0.4
Master        : local[*]
Parallelism   : 2


## 2. Giới thiệu Data Cleaning

Data Cleaning là quá trình chuẩn bị dữ liệu thô trước khi đưa vào pipeline xử lý.

Các công việc thường gặp:

- chuẩn hóa hoặc thay thế chuỗi;
- tính toán lại giá trị;
- loại dữ liệu rác;
- xử lý dữ liệu thiếu;
- kiểm tra kiểu dữ liệu;
- chuẩn hóa định dạng;
- xác thực dữ liệu.

Spark phù hợp cho data cleaning khi dữ liệu có kích thước lớn vì có khả năng xử lý phân tán và mở rộng.

### 2.1. Ví dụ làm sạch dữ liệu đơn giản

Dữ liệu thô:

| name | age (years) | city |
|---|---:|---|
| Smith, John | 37 | Dallas |
| Wilson, A. | 59 | Chicago |
| null | 215 | null |

Mục tiêu:

- tách họ và tên;
- đổi tuổi từ năm sang tháng;
- đổi tên thành phố sang mã bang;
- loại dòng không hợp lệ.

In [19]:
raw_data = [
    ("Smith, John", 37, "Dallas"),
    ("Wilson, A.", 59, "Chicago"),
    (None, 215, None)
]

raw_df = spark.createDataFrame(raw_data, ["name", "age_years", "city"])
raw_df.show(truncate=False)

+-----------+---------+-------+
|name       |age_years|city   |
+-----------+---------+-------+
|Smith, John|37       |Dallas |
|Wilson, A. |59       |Chicago|
|NULL       |215      |NULL   |
+-----------+---------+-------+



In [20]:
city_to_state = (
    F.when(F.col("city") == "Dallas", "TX")
     .when(F.col("city") == "Chicago", "IL")
)

clean_demo_df = (
    raw_df
    .filter(F.col("name").isNotNull())
    .withColumn("last_name", F.trim(F.split("name", ",").getItem(0)))
    .withColumn("first_name", F.trim(F.split("name", ",").getItem(1)))
    .withColumn("age_months", F.col("age_years") * 12)
    .withColumn("state", city_to_state)
    .select("last_name", "first_name", "age_months", "state")
)

clean_demo_df.show(truncate=False)

+---------+----------+----------+-----+
|last_name|first_name|age_months|state|
+---------+----------+----------+-----+
|Smith    |John      |444       |TX   |
|Wilson   |A.        |708       |IL   |
+---------+----------+----------+-----+



<a id="schema"></a>
## 3. Spark Schema

Schema mô tả cấu trúc của DataFrame:

- tên cột;
- kiểu dữ liệu;
- có cho phép `null` hay không.

Việc khai báo schema rõ ràng giúp:

- phát hiện dữ liệu sai kiểu sớm hơn;
- tránh Spark phải quét dữ liệu nhiều lần để suy luận kiểu;
- tăng tính nhất quán trong pipeline.

### 3.1. Định nghĩa schema bằng `StructType`

In [21]:
people_schema = StructType([
    StructField("name", StringType(), False),
    StructField("age", IntegerType(), False),
    StructField("city", StringType(), False)
])

print(people_schema.simpleString())

struct<name:string,age:int,city:string>


### 3.2. Ví dụ schema cho dữ liệu chuyến bay

Dataset trong ZIP gồm dữ liệu chuyến bay American Airlines từ DFW.

In [22]:
flight_schema = StructType([
    StructField("Date (MM/DD/YYYY)", StringType(), True),
    StructField("Flight Number", StringType(), True),
    StructField("Destination Airport", StringType(), True),
    StructField("Actual elapsed time (Minutes)", IntegerType(), True)
])

In [23]:
if DATA_DIR is not None:
    flight_2014 = spark.read.csv(
        str(DATA_DIR / "AA_DFW_2014_Departures_Short.csv.gz"),
        header=True,
        schema=flight_schema
    )

    flight_2014.printSchema()
    flight_2014.show(5)

root
 |-- Date (MM/DD/YYYY): string (nullable = true)
 |-- Flight Number: string (nullable = true)
 |-- Destination Airport: string (nullable = true)
 |-- Actual elapsed time (Minutes): integer (nullable = true)

+-----------------+-------------+-------------------+-----------------------------+
|Date (MM/DD/YYYY)|Flight Number|Destination Airport|Actual elapsed time (Minutes)|
+-----------------+-------------+-------------------+-----------------------------+
|       01/01/2014|         0005|                HNL|                          519|
|       01/01/2014|         0007|                OGG|                          505|
|       01/01/2014|         0035|                SLC|                          174|
|       01/01/2014|         0043|                DTW|                          153|
|       01/01/2014|         0052|                PIT|                          137|
+-----------------+-------------+-------------------+-----------------------------+
only showing top 5 rows


### So sánh với `inferSchema=True`

`inferSchema=True` tiện lợi trong ví dụ nhỏ, nhưng trên dữ liệu lớn Spark phải đọc thêm dữ liệu để suy luận kiểu.

Trong các pipeline production, nên ưu tiên schema được định nghĩa trước nếu cấu trúc dữ liệu đã biết.

## 4. Immutability và Lazy Processing

### 4.1. Immutability

Spark DataFrame là immutable, không sửa trực tiếp DataFrame gốc.

Mỗi transformation tạo ra một DataFrame mới về mặt logic.

In [24]:
df_original = spark.createDataFrame([
    ("Alice", 24),
    ("Bob", 30)
], ["name", "age"])

df_new = df_original.withColumn("age_next_year", F.col("age") + 1)

print("DataFrame gốc:")
df_original.show()

print("DataFrame mới:")
df_new.show()

DataFrame gốc:
+-----+---+
| name|age|
+-----+---+
|Alice| 24|
|  Bob| 30|
+-----+---+

DataFrame mới:
+-----+---+-------------+
| name|age|age_next_year|
+-----+---+-------------+
|Alice| 24|           25|
|  Bob| 30|           31|
+-----+---+-------------+



### 4.2. Lazy Processing

Các transformation như:

- `withColumn()`
- `select()`
- `filter()`
- `drop()`

chưa thực thi ngay.

Spark chỉ thật sự xử lý khi gặp **action** như:

- `show()`
- `count()`
- `collect()`
- `write`

In [25]:
if DATA_DIR is not None:
    lazy_df = spark.read.csv(
        str(DATA_DIR / "AA_DFW_2015_Departures_Short.csv.gz"),
        header=True,
        inferSchema=True
    )

    lazy_df = lazy_df.withColumn(
        "airport",
        F.lower(F.col("Destination Airport"))
    )

    lazy_df = lazy_df.drop("Destination Airport")

    # Action:
    lazy_df.show(5)

+-----------------+-------------+-----------------------------+-------+
|Date (MM/DD/YYYY)|Flight Number|Actual elapsed time (Minutes)|airport|
+-----------------+-------------+-----------------------------+-------+
|       01/01/2015|            5|                          526|    hnl|
|       01/01/2015|            7|                          517|    ogg|
|       01/01/2015|           23|                          233|    sfo|
|       01/01/2015|           27|                          165|    las|
|       01/01/2015|           29|                            0|    ont|
+-----------------+-------------+-----------------------------+-------+
only showing top 5 rows


### 4.3. Xem Logical/Physical Plan

`explain()` giúp quan sát kế hoạch Spark dự kiến thực hiện.

In [26]:
if DATA_DIR is not None:
    lazy_df.filter(F.col("airport") == "hnl").select("airport").explain()

== Physical Plan ==
*(1) Project [lower(Destination Airport#104) AS airport#107]
+- *(1) Filter (isnotnull(Destination Airport#104) AND (lower(Destination Airport#104) = hnl))
   +- FileScan csv [Destination Airport#104] Batched: false, DataFilters: [isnotnull(Destination Airport#104), (lower(Destination Airport#104) = hnl)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/Datasets/AA_DFW_2015_Departures_Short.csv.gz], PartitionFilters: [], PushedFilters: [IsNotNull(Destination Airport)], ReadSchema: struct<Destination Airport:string>




## 5. Làm việc với Parquet

CSV dễ trao đổi nhưng có các hạn chế:

- không lưu schema;
- cần parse chuỗi;
- kém hiệu quả với dữ liệu lớn;
- hạn chế predicate pushdown.

Parquet là định dạng dạng cột (columnar), phù hợp với Spark vì:

- lưu schema;
- hỗ trợ predicate pushdown;
- đọc chọn lọc cột hiệu quả;
- thường tốt hơn cho phân tích.

### 5.1. Gộp nhiều năm dữ liệu

In [27]:
if DATA_DIR is not None:
    files = [
        "AA_DFW_2014_Departures_Short.csv.gz",
        "AA_DFW_2015_Departures_Short.csv.gz",
        "AA_DFW_2016_Departures_Short.csv.gz",
        "AA_DFW_2017_Departures_Short.csv.gz"
    ]

    flight_dfs = [
        spark.read.csv(str(DATA_DIR / f), header=True, schema=flight_schema)
        for f in files
    ]

    flights_all = flight_dfs[0]
    for df in flight_dfs[1:]:
        flights_all = flights_all.unionByName(df)

    print("Tổng số dòng:", flights_all.count())
    flights_all.show(5)

Tổng số dòng: 583718
+-----------------+-------------+-------------------+-----------------------------+
|Date (MM/DD/YYYY)|Flight Number|Destination Airport|Actual elapsed time (Minutes)|
+-----------------+-------------+-------------------+-----------------------------+
|       01/01/2014|         0005|                HNL|                          519|
|       01/01/2014|         0007|                OGG|                          505|
|       01/01/2014|         0035|                SLC|                          174|
|       01/01/2014|         0043|                DTW|                          153|
|       01/01/2014|         0052|                PIT|                          137|
+-----------------+-------------+-------------------+-----------------------------+
only showing top 5 rows


### 5.2. Ghi và đọc lại Parquet

In [28]:
if DATA_DIR is not None:
    PARQUET_PATH = str(Path.cwd() / "AA_DFW_ALL.parquet")

    (
        flights_all.write
        .mode("overwrite")
        .parquet(PARQUET_PATH)
    )

    flights_parquet = spark.read.parquet(PARQUET_PATH)

    print("Đọc lại từ Parquet:", flights_parquet.count())
    flights_parquet.printSchema()

Đọc lại từ Parquet: 583718
root
 |-- Date (MM/DD/YYYY): string (nullable = true)
 |-- Flight Number: string (nullable = true)
 |-- Destination Airport: string (nullable = true)
 |-- Actual elapsed time (Minutes): integer (nullable = true)



### 5.3. Parquet và Spark SQL

In [29]:
if DATA_DIR is not None:
    flights_parquet.createOrReplaceTempView("flights")

    spark.sql('''
        SELECT
            `Destination Airport` AS airport,
            AVG(`Actual elapsed time (Minutes)`) AS avg_minutes,
            COUNT(*) AS n
        FROM flights
        GROUP BY `Destination Airport`
        ORDER BY n DESC
        LIMIT 10
    ''').show()

+-------+------------------+-----+
|airport|       avg_minutes|    n|
+-------+------------------+-----+
|    LAX|195.81170700980834|22226|
|    ORD| 134.2056630529253|21861|
|    SAT|63.446708969670894|18596|
|    AUS|  56.0158876979161|18379|
|    LGA|191.42767295597486|17013|
|    MIA|161.56616822429908|16050|
|    LAS|170.62809139784946|14880|
|    SFO|222.81960865657194|14463|
|    PHX| 149.9250647804468|14279|
|    MCO|148.37622832369942|13840|
+-------+------------------+-----+



## 6. DataFrame Column Operations

Các transformation phổ biến:

- `filter()` / `where()`
- `select()`
- `withColumn()`
- `drop()`
- `withColumnRenamed()`

### 6.1. Dataset Dallas Council Voters

In [30]:
if DATA_DIR is not None:
    voter_df = spark.read.csv(
        str(DATA_DIR / "DallasCouncilVoters.csv.gz"),
        header=True,
        inferSchema=True
    )

    voter_df.printSchema()
    voter_df.show(10, truncate=False)

root
 |-- DATE: string (nullable = true)
 |-- TITLE: string (nullable = true)
 |-- VOTER_NAME: string (nullable = true)

+----------+-------------+-------------------+
|DATE      |TITLE        |VOTER_NAME         |
+----------+-------------+-------------------+
|02/08/2017|Councilmember|Jennifer S. Gates  |
|02/08/2017|Councilmember|Philip T. Kingston |
|02/08/2017|Mayor        |Michael S. Rawlings|
|02/08/2017|Councilmember|Adam Medrano       |
|02/08/2017|Councilmember|Casey Thomas       |
|02/08/2017|Councilmember|Carolyn King Arnold|
|02/08/2017|Councilmember|Scott Griggs       |
|02/08/2017|Councilmember|B. Adam  McGough   |
|02/08/2017|Councilmember|Lee Kleinman       |
|02/08/2017|Councilmember|Sandy Greyson      |
+----------+-------------+-------------------+
only showing top 10 rows


### 6.2. Filtering

In [31]:
if DATA_DIR is not None:
    voter_df.select("VOTER_NAME").distinct().show(20, truncate=False)

    voter_clean = (
        voter_df
        .filter(F.col("VOTER_NAME").isNotNull())
        .filter(F.length("VOTER_NAME").between(1, 20))
        .filter(~F.col("VOTER_NAME").contains("_"))
    )

    voter_clean.select("VOTER_NAME").distinct().show(20, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|VOTER_NAME                                                                                                                                                                                                                                                                                                                                                                                                                |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### 6.3. String transformations

Các hàm thường dùng trong `pyspark.sql.functions`:

- `upper()`, `lower()`
- `trim()`
- `split()`
- `length()`
- `contains()`
- `regexp_replace()`

In [32]:
if DATA_DIR is not None:
    voter_string = (
        voter_df
        .filter(F.col("VOTER_NAME").isNotNull())
        .withColumn("NAME_UPPER", F.upper("VOTER_NAME"))
        .withColumn("NAME_PARTS", F.split("VOTER_NAME", " "))
        .withColumn("PART_COUNT", F.size("NAME_PARTS"))
    )

    voter_string.select(
        "VOTER_NAME", "NAME_UPPER", "NAME_PARTS", "PART_COUNT"
    ).show(10, truncate=False)

+-------------------+-------------------+-----------------------+----------+
|VOTER_NAME         |NAME_UPPER         |NAME_PARTS             |PART_COUNT|
+-------------------+-------------------+-----------------------+----------+
|Jennifer S. Gates  |JENNIFER S. GATES  |[Jennifer, S., Gates]  |3         |
|Philip T. Kingston |PHILIP T. KINGSTON |[Philip, T., Kingston] |3         |
|Michael S. Rawlings|MICHAEL S. RAWLINGS|[Michael, S., Rawlings]|3         |
|Adam Medrano       |ADAM MEDRANO       |[Adam, Medrano]        |2         |
|Casey Thomas       |CASEY THOMAS       |[Casey, Thomas]        |2         |
|Carolyn King Arnold|CAROLYN KING ARNOLD|[Carolyn, King, Arnold]|3         |
|Scott Griggs       |SCOTT GRIGGS       |[Scott, Griggs]        |2         |
|B. Adam  McGough   |B. ADAM  MCGOUGH   |[B., Adam, , McGough]  |4         |
|Lee Kleinman       |LEE KLEINMAN       |[Lee, Kleinman]        |2         |
|Sandy Greyson      |SANDY GREYSON      |[Sandy, Greyson]       |2         |

### 6.4. Tách first name / last name

[MỞ RỘNG] Code trong ZIP tách tên từ dữ liệu Dallas Council.

In [33]:
if DATA_DIR is not None:
    voter_names = (
        voter_df
        .filter(F.col("VOTER_NAME").isNotNull())
        .withColumn("name_parts", F.split(F.trim("VOTER_NAME"), " "))
        .withColumn("first_name", F.col("name_parts").getItem(0))
        .withColumn(
            "last_name",
            F.col("name_parts").getItem(
                F.size("name_parts") - 1
            )
        )
    )

    voter_names.select(
        "VOTER_NAME", "first_name", "last_name"
    ).show(10, truncate=False)

/usr/local/lib/python3.13/dist-packages/pyspark/sql/classic/column.py:359: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(


+-------------------+----------+---------+
|VOTER_NAME         |first_name|last_name|
+-------------------+----------+---------+
|Jennifer S. Gates  |Jennifer  |Gates    |
|Philip T. Kingston |Philip    |Kingston |
|Michael S. Rawlings|Michael   |Rawlings |
|Adam Medrano       |Adam      |Medrano  |
|Casey Thomas       |Casey     |Thomas   |
|Carolyn King Arnold|Carolyn   |Arnold   |
|Scott Griggs       |Scott     |Griggs   |
|B. Adam  McGough   |B.        |McGough  |
|Lee Kleinman       |Lee       |Kleinman |
|Sandy Greyson      |Sandy     |Greyson  |
+-------------------+----------+---------+
only showing top 10 rows


## 7. Conditional DataFrame Column Operations

Spark hỗ trợ logic tương tự `if / elif / else` qua:

- `F.when()`
- `.otherwise()`

In [34]:
age_df = spark.createDataFrame([
    ("Alice", 14),
    ("Bob", 18),
    ("Candice", 38)
], ["Name", "Age"])

classified_df = age_df.withColumn(
    "Age_Group",
    F.when(F.col("Age") >= 18, "Adult")
     .otherwise("Minor")
)

classified_df.show()

+-------+---+---------+
|   Name|Age|Age_Group|
+-------+---+---------+
|  Alice| 14|    Minor|
|    Bob| 18|    Adult|
|Candice| 38|    Adult|
+-------+---+---------+



### 7.1. Nhiều điều kiện

In [35]:
multi_df = age_df.withColumn(
    "Category",
    F.when(F.col("Age") < 13, "Child")
     .when(F.col("Age") < 18, "Teen")
     .otherwise("Adult")
)

multi_df.show()

+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 14|    Teen|
|    Bob| 18|   Adult|
|Candice| 38|   Adult|
+-------+---+--------+



## 8. User Defined Functions - UDF

UDF cho phép dùng hàm Python tùy biến trong DataFrame.

Quy trình:

1. viết hàm Python;
2. bọc hàm bằng `F.udf`;
3. khai báo kiểu dữ liệu trả về;
4. dùng như một hàm Spark.

> Trong thực tế, nếu có thể biểu diễn bằng built-in Spark functions thì nên ưu tiên built-in functions vì thường tối ưu hơn Python UDF.

### 8.1. UDF đảo chuỗi

In [36]:
def reverse_string(text):
    if text is None:
        return None
    return text[::-1]

udf_reverse = F.udf(reverse_string, StringType())

if DATA_DIR is not None:
    voter_udf = voter_df.withColumn(
        "ReverseName",
        udf_reverse("VOTER_NAME")
    )

    voter_udf.select("VOTER_NAME", "ReverseName").show(10, truncate=False)

+-------------------+-------------------+
|VOTER_NAME         |ReverseName        |
+-------------------+-------------------+
|Jennifer S. Gates  |setaG .S refinneJ  |
|Philip T. Kingston |notsgniK .T pilihP |
|Michael S. Rawlings|sgnilwaR .S leahciM|
|Adam Medrano       |onardeM madA       |
|Casey Thomas       |samohT yesaC       |
|Carolyn King Arnold|dlonrA gniK nyloraC|
|Scott Griggs       |sggirG ttocS       |
|B. Adam  McGough   |hguoGcM  madA .B   |
|Lee Kleinman       |namnielK eeL       |
|Sandy Greyson      |nosyerG ydnaS      |
+-------------------+-------------------+
only showing top 10 rows


## 8.2. [MỞ RỘNG] UDF tính toán

In [37]:
def name_word_count(name):
    if name is None:
        return 0
    return len(name.split())

udf_name_word_count = F.udf(name_word_count, IntegerType())

if DATA_DIR is not None:
    voter_udf = voter_udf.withColumn(
        "NameWordCount",
        udf_name_word_count("VOTER_NAME")
    )

    voter_udf.select(
        "VOTER_NAME", "NameWordCount"
    ).show(10, truncate=False)

+-------------------+-------------+
|VOTER_NAME         |NameWordCount|
+-------------------+-------------+
|Jennifer S. Gates  |3            |
|Philip T. Kingston |3            |
|Michael S. Rawlings|3            |
|Adam Medrano       |2            |
|Casey Thomas       |2            |
|Carolyn King Arnold|3            |
|Scott Griggs       |2            |
|B. Adam  McGough   |3            |
|Lee Kleinman       |2            |
|Sandy Greyson      |2            |
+-------------------+-------------+
only showing top 10 rows


<a id="partition"></a>
# 9. Partitioning và ID

Spark chia DataFrame thành nhiều **partition**.  
Mỗi partition có thể được xử lý độc lập trên các worker.

Điều này giúp xử lý song song, nhưng cũng khiến việc tạo ID tuần tự `0,1,2,...` không tự nhiên trong môi trường phân tán.

## 9.1. Quan sát số partition

In [38]:
if DATA_DIR is not None:
    print("Partitions:", voter_df.rdd.getNumPartitions())

Partitions: 1


## 9.2. `monotonically_increasing_id()`

Hàm này tạo ID:

- 64-bit;
- duy nhất;
- tăng dần về mặt giá trị;
- **không đảm bảo liên tục**.

In [39]:
if DATA_DIR is not None:
    voter_with_id = voter_df.withColumn(
        "ID",
        F.monotonically_increasing_id()
    )

    voter_with_id.select("ID", "VOTER_NAME").show(10, truncate=False)

+---+-------------------+
|ID |VOTER_NAME         |
+---+-------------------+
|0  |Jennifer S. Gates  |
|1  |Philip T. Kingston |
|2  |Michael S. Rawlings|
|3  |Adam Medrano       |
|4  |Casey Thomas       |
|5  |Carolyn King Arnold|
|6  |Scott Griggs       |
|7  |B. Adam  McGough   |
|8  |Lee Kleinman       |
|9  |Sandy Greyson      |
+---+-------------------+
only showing top 10 rows


## 9.3. ID và Lazy Processing

Vì Spark lazy và partition-aware, ID có thể thay đổi khi kế hoạch thực thi thay đổi.  
Không nên xem `monotonically_increasing_id()` như primary key tuần tự của database.

## 10. Caching

Caching lưu DataFrame ở memory/disk để tái sử dụng nhanh hơn.

Phù hợp khi:

- một DataFrame được dùng nhiều lần;
- các bước trước đó tốn nhiều chi phí;
- dữ liệu đủ phù hợp với tài nguyên hệ thống.

Không nên cache mọi thứ.

### 10.1. Cache DataFrame

In [40]:
if DATA_DIR is not None:
    cached_df = voter_df.cache()

    # Action để materialize cache
    row_count = cached_df.count()

    print("Rows:", row_count)
    print("Is cached:", cached_df.is_cached)

Rows: 44625
Is cached: True


### 10.2. Unpersist

In [41]:
if DATA_DIR is not None:
    cached_df.unpersist()
    print("Is cached:", cached_df.is_cached)

Is cached: False


## 11. Cải thiện hiệu năng import

Một số nguyên tắc trong slide:

- định nghĩa schema rõ ràng;
- dữ liệu nên được chia thành các file có kích thước tương đối cân bằng;
- có thể đọc nhiều file bằng wildcard;
- dữ liệu trung gian nên chuyển sang Parquet nếu phù hợp.

### 11.1. Đọc nhiều file chuyến bay bằng wildcard

In [42]:
if DATA_DIR is not None:
    wildcard_path = str(DATA_DIR / "AA_DFW_*_Departures_Short.csv.gz")

    flights_wildcard = spark.read.csv(
        wildcard_path,
        header=True,
        schema=flight_schema
    )

    print("Số dòng:", flights_wildcard.count())

Số dòng: 583718


### 11.2. So sánh schema explicit và inferSchema

In [43]:
if DATA_DIR is not None:
    explicit_df = spark.read.csv(
        wildcard_path,
        header=True,
        schema=flight_schema
    )

    inferred_df = spark.read.csv(
        wildcard_path,
        header=True,
        inferSchema=True
    )

    print("Explicit schema:")
    explicit_df.printSchema()

    print("Inferred schema:")
    inferred_df.printSchema()

Explicit schema:
root
 |-- Date (MM/DD/YYYY): string (nullable = true)
 |-- Flight Number: string (nullable = true)
 |-- Destination Airport: string (nullable = true)
 |-- Actual elapsed time (Minutes): integer (nullable = true)

Inferred schema:
root
 |-- Date (MM/DD/YYYY): string (nullable = true)
 |-- Flight Number: integer (nullable = true)
 |-- Destination Airport: string (nullable = true)
 |-- Actual elapsed time (Minutes): integer (nullable = true)



### 11.3 Spark Configuration và Cluster

Spark có nhiều cấu hình có thể đọc/điều chỉnh qua:

```python
spark.conf.get(...)
spark.conf.set(...)
```

Trong notebook local, ta chỉ quan sát một số cấu hình an toàn.

In [44]:
print("spark.sql.shuffle.partitions =", spark.conf.get("spark.sql.shuffle.partitions"))
print("spark.sql.adaptive.enabled   =", spark.conf.get("spark.sql.adaptive.enabled"))

spark.sql.shuffle.partitions = 200
spark.sql.adaptive.enabled   = true


### Kiến trúc cluster

Hai nhóm process quan trọng:

### Driver
- lập kế hoạch công việc;
- phân phối task;
- tổng hợp kết quả.

### Worker
- thực thi task;
- xử lý partition;
- đọc/ghi dữ liệu.

## 12. Execution Plan, Shuffle và Broadcast

### 12.1. Execution Plan

`explain()` giúp xem Spark sẽ thực hiện truy vấn như thế nào.

In [45]:
if DATA_DIR is not None:
    voter_df.select("VOTER_NAME").distinct().explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[VOTER_NAME#265], functions=[])
   +- Exchange hashpartitioning(VOTER_NAME#265, 200), ENSURE_REQUIREMENTS, [plan_id=733]
      +- HashAggregate(keys=[VOTER_NAME#265], functions=[])
         +- FileScan csv [VOTER_NAME#265] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/Datasets/DallasCouncilVoters.csv.gz], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<VOTER_NAME:string>




### 12.2. Shuffle

Shuffle xảy ra khi Spark phải di chuyển dữ liệu giữa các partition/worker.

Các thao tác dễ phát sinh shuffle:

- `groupBy`
- `distinct`
- `join`
- `repartition`

Shuffle thường tốn network và I/O.

### 12.3. `coalesce()` và `repartition()`

- `repartition(n)`: có thể tăng/giảm partition, thường gây shuffle.
- `coalesce(n)`: chủ yếu dùng để giảm số partition với ít shuffle hơn.

In [46]:
if DATA_DIR is not None:
    print("Ban đầu:", flights_wildcard.rdd.getNumPartitions())

    repart_df = flights_wildcard.repartition(4)
    print("repartition(4):", repart_df.rdd.getNumPartitions())

    coalesce_df = repart_df.coalesce(2)
    print("coalesce(2):", coalesce_df.rdd.getNumPartitions())

Ban đầu: 2
repartition(4): 4
coalesce(2): 2


## 12.4. Broadcast Join

Nếu một DataFrame nhỏ, Spark có thể gửi bản sao tới các worker để giảm shuffle.

In [47]:
airport_info = spark.createDataFrame([
    ("HNL", "Honolulu"),
    ("OGG", "Kahului"),
    ("SFO", "San Francisco"),
    ("LAS", "Las Vegas"),
    ("DTW", "Detroit")
], ["airport_code", "airport_name"])

if DATA_DIR is not None:
    joined_broadcast = flights_wildcard.join(
        F.broadcast(airport_info),
        flights_wildcard["Destination Airport"] == airport_info["airport_code"],
        "left"
    )

    joined_broadcast.select(
        "Destination Airport", "airport_name"
    ).show(10)

+-------------------+------------+
|Destination Airport|airport_name|
+-------------------+------------+
|                HNL|    Honolulu|
|                OGG|     Kahului|
|                SLC|        NULL|
|                DTW|     Detroit|
|                PIT|        NULL|
|                SAN|        NULL|
|                MIA|        NULL|
|                JFK|        NULL|
|                ORD|        NULL|
|                STL|        NULL|
+-------------------+------------+
only showing top 10 rows


## 13. Data Pipeline

Một data pipeline gồm:

```text
Input
  ↓
Validation
  ↓
Cleaning
  ↓
Transformation
  ↓
Analysis
  ↓
Output
```

Nguồn có thể là:

- CSV
- JSON
- Parquet
- Database
- Web service

Output có thể là:

- Parquet
- CSV
- Database
- Dashboard / BI

### 13.1. Pipeline mẫu cho Dallas Council Votes

Ta sẽ:

1. đọc dữ liệu;
2. chuẩn hóa tên cột;
3. loại voter null;
4. chuẩn hóa chữ;
5. thêm ID;
6. tạo bảng phân tích.

In [48]:
if DATA_DIR is not None:
    votes_df = spark.read.csv(
        str(DATA_DIR / "DallasCouncilVotes.csv.gz"),
        header=True,
        inferSchema=True
    )

    votes_df.printSchema()

root
 |-- DATE: string (nullable = true)
 |-- AGENDA_ITEM_NUMBER: string (nullable = true)
 |-- ITEM_TYPE: string (nullable = true)
 |-- DISTRICT: string (nullable = true)
 |-- TITLE: string (nullable = true)
 |-- VOTER NAME: string (nullable = true)
 |-- VOTE CAST: string (nullable = true)
 |-- FINAL ACTION TAKEN: string (nullable = true)
 |-- AGENDA ITEM DESCRIPTION: string (nullable = true)
 |-- AGENDA_ID: string (nullable = true)
 |-- VOTE_ID: string (nullable = true)



In [49]:
if DATA_DIR is not None:
    votes_clean = (
        votes_df
        .withColumnRenamed("VOTER NAME", "VOTER_NAME")
        .withColumnRenamed("VOTE CAST", "VOTE_CAST")
        .filter(F.col("VOTER_NAME").isNotNull())
        .withColumn("VOTER_NAME", F.trim(F.col("VOTER_NAME")))
        .withColumn("VOTER_NAME_UPPER", F.upper("VOTER_NAME"))
        .withColumn("ROW_ID", F.monotonically_increasing_id())
    )

    votes_clean.select(
        "ROW_ID", "DATE", "VOTER_NAME", "VOTE_CAST"
    ).show(10, truncate=False)

+------+----------+-------------------+---------+
|ROW_ID|DATE      |VOTER_NAME         |VOTE_CAST|
+------+----------+-------------------+---------+
|0     |02/08/2017|Jennifer S. Gates  |N/A      |
|1     |02/08/2017|Philip T. Kingston |N/A      |
|2     |02/08/2017|Michael S. Rawlings|N/A      |
|3     |02/08/2017|Adam Medrano       |N/A      |
|4     |02/08/2017|Casey Thomas, II   |N/A      |
|5     |02/08/2017|Carolyn King Arnold|N/A      |
|6     |02/08/2017|Scott Griggs       |YES      |
|7     |02/08/2017|B. Adam  McGough   |YES      |
|8     |02/08/2017|Lee Kleinman       |YES      |
|9     |02/08/2017|Sandy Greyson      |YES      |
+------+----------+-------------------+---------+
only showing top 10 rows


### 13.2. Phân tích sau cleaning

In [50]:
if DATA_DIR is not None:
    (
        votes_clean
        .groupBy("VOTE_CAST")
        .count()
        .orderBy(F.desc("count"))
        .show(truncate=False)
    )

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|VOTE_CAST                                                                                                                                                                                                                                        

### 13.3. Xuất Parquet

In [51]:
if DATA_DIR is not None:
    OUTPUT_PATH = str(Path.cwd() / "DallasCouncilVotes_Clean.parquet")

    (
        votes_clean.write
        .mode("overwrite")
        .parquet(OUTPUT_PATH)
    )

    print("Đã ghi:", OUTPUT_PATH)

Đã ghi: /content/DallasCouncilVotes_Clean.parquet


## 14. Data Handling và Validation

Dữ liệu CSV thực tế có thể chứa:

- dòng trống;
- comment;
- header;
- số cột không đồng nhất;
- nhiều delimiter;
- cấu trúc lồng nhau;
- dữ liệu sai kiểu.

Spark hỗ trợ nhiều tùy chọn khi đọc CSV.

### Ví dụ comment và separator

In [52]:
from tempfile import TemporaryDirectory
from pathlib import Path

tmp_dir = TemporaryDirectory()
sample_file = Path(tmp_dir.name) / "sample.csv"

sample_file.write_text(
    "# Dòng comment\n"
    "name;age;city\n"
    "Alice;23;HCMC\n"
    "Bob;31;Hanoi\n",
    encoding="utf-8"
)

sample_df = spark.read.csv(
    str(sample_file),
    header=True,
    sep=";",
    comment="#",
    inferSchema=True
)

sample_df.show()

+-----+---+-----+
| name|age| city|
+-----+---+-----+
|Alice| 23| HCMC|
|  Bob| 31|Hanoi|
+-----+---+-----+



### Data Validation

Validation là kiểm tra dữ liệu có phù hợp với format/rule mong đợi không.

Có thể kiểm tra:

- số dòng;
- số cột;
- kiểu dữ liệu;
- miền giá trị;
- khóa hợp lệ;
- quan hệ với bảng tham chiếu.

### 14.2. Validation bằng join

In [53]:
valid_titles = spark.createDataFrame([
    ("Mayor",),
    ("Councilmember",)
], ["TITLE"])

if DATA_DIR is not None:
    before = voter_df.count()

    verified_voters = voter_df.join(
        F.broadcast(valid_titles),
        on="TITLE",
        how="inner"
    )

    after = verified_voters.count()

    print("Before:", before)
    print("After :", after)

Before: 44625
After : 38020


### 14.2. Tìm dữ liệu không hợp lệ bằng `left_anti`

In [54]:
if DATA_DIR is not None:
    invalid_voters = voter_df.join(
        F.broadcast(valid_titles),
        on="TITLE",
        how="left_anti"
    )

    print("Invalid rows:", invalid_voters.count())
    invalid_voters.show(10, truncate=False)

Invalid rows: 6605
+--------------------+----------+-----------------+
|TITLE               |DATE      |VOTER_NAME       |
+--------------------+----------+-----------------+
|Deputy Mayor Pro Tem|04/25/2018|Adam Medrano     |
|Mayor Pro Tem       |04/25/2018|Dwaine R. Caraway|
|Deputy Mayor Pro Tem|06/20/2018|Adam Medrano     |
|Mayor Pro Tem       |06/20/2018|Dwaine R. Caraway|
|Deputy Mayor Pro Tem|06/20/2018|Adam Medrano     |
|Mayor Pro Tem       |06/20/2018|Dwaine R. Caraway|
|Deputy Mayor Pro Tem|08/15/2018|Adam Medrano     |
|Vacant              |08/15/2018|NULL             |
|Deputy Mayor Pro Tem|08/15/2018|Adam Medrano     |
|Vacant              |08/15/2018|NULL             |
+--------------------+----------+-----------------+
only showing top 10 rows


## 14.3. [MỞ RỘNG] Pipeline xử lý annotation dạng không chuẩn

Code ZIP có một ví dụ mở rộng với Stanford ImageNet dog annotations:

- tách nhiều trường từ một dòng;
- xử lý `ArrayType`;
- dùng UDF để chuyển chuỗi thành cấu trúc;
- validate folder bằng join;
- dùng `left_anti` để tìm dòng invalid;
- tính phần trăm pixel thuộc bounding box của chó.

**Dataset annotation không được đóng gói trong ZIP**, nên phần dưới dùng dữ liệu mô phỏng nhỏ để minh họa đúng kỹ thuật.

In [55]:
annotation_data = [
    ("02111277 n02111277_3206 500 375 Newfoundland,110,73,416,298",),
    ("02108422 n02108422_4375 500 375 bull_mastiff,101,90,214,356 bull_mastiff,282,74,416,370",)
]

annotations_df = spark.createDataFrame(annotation_data, ["raw"])
annotations_df.show(truncate=False)

+---------------------------------------------------------------------------------------+
|raw                                                                                    |
+---------------------------------------------------------------------------------------+
|02111277 n02111277_3206 500 375 Newfoundland,110,73,416,298                            |
|02108422 n02108422_4375 500 375 bull_mastiff,101,90,214,356 bull_mastiff,282,74,416,370|
+---------------------------------------------------------------------------------------+



In [56]:
split_cols = F.split(F.col("raw"), " ")

parsed = (
    annotations_df
    .withColumn("folder", split_cols.getItem(0))
    .withColumn("filename", split_cols.getItem(1))
    .withColumn("width", split_cols.getItem(2).cast("int"))
    .withColumn("height", split_cols.getItem(3).cast("int"))
    .withColumn("parts", split_cols)
)

parsed.select("folder", "filename", "width", "height", "parts").show(truncate=False)

+--------+--------------+-----+------+----------------------------------------------------------------------------------------------+
|folder  |filename      |width|height|parts                                                                                         |
+--------+--------------+-----+------+----------------------------------------------------------------------------------------------+
|02111277|n02111277_3206|500  |375   |[02111277, n02111277_3206, 500, 375, Newfoundland,110,73,416,298]                             |
|02108422|n02108422_4375|500  |375   |[02108422, n02108422_4375, 500, 375, bull_mastiff,101,90,214,356, bull_mastiff,282,74,416,370]|
+--------+--------------+-----+------+----------------------------------------------------------------------------------------------+



## 14. Bài tập mở rộng: Làm sạch dữ liệu chuyến bay

### Yêu cầu

Từ 4 file chuyến bay 2014-2017:

1. đọc bằng schema cố định;
2. gộp dữ liệu;
3. chuẩn hóa tên cột;
4. loại thời gian bay null hoặc <= 0;
5. thêm cột `year`;
6. chuẩn hóa airport thành uppercase;
7. tính thống kê theo năm và destination;
8. lưu kết quả thành Parquet.

In [57]:
if DATA_DIR is not None:
    clean_flights = (
        flights_wildcard
        .withColumnRenamed("Date (MM/DD/YYYY)", "flight_date")
        .withColumnRenamed("Flight Number", "flight_number")
        .withColumnRenamed("Destination Airport", "destination")
        .withColumnRenamed(
            "Actual elapsed time (Minutes)",
            "elapsed_minutes"
        )
        .filter(F.col("elapsed_minutes").isNotNull())
        .filter(F.col("elapsed_minutes") > 0)
        .withColumn("destination", F.upper(F.trim("destination")))
        .withColumn(
            "flight_date_parsed",
            F.to_date("flight_date", "MM/dd/yyyy")
        )
        .withColumn("year", F.year("flight_date_parsed"))
    )

    clean_flights.printSchema()
    clean_flights.show(10)

root
 |-- flight_date: string (nullable = true)
 |-- flight_number: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- elapsed_minutes: integer (nullable = true)
 |-- flight_date_parsed: date (nullable = true)
 |-- year: integer (nullable = true)

+-----------+-------------+-----------+---------------+------------------+----+
|flight_date|flight_number|destination|elapsed_minutes|flight_date_parsed|year|
+-----------+-------------+-----------+---------------+------------------+----+
| 01/01/2014|         0005|        HNL|            519|        2014-01-01|2014|
| 01/01/2014|         0007|        OGG|            505|        2014-01-01|2014|
| 01/01/2014|         0035|        SLC|            174|        2014-01-01|2014|
| 01/01/2014|         0043|        DTW|            153|        2014-01-01|2014|
| 01/01/2014|         0052|        PIT|            137|        2014-01-01|2014|
| 01/01/2014|         0058|        SAN|            174|        2014-01-01|2014|
| 01/01/20

In [58]:
if DATA_DIR is not None:
    flight_stats = (
        clean_flights
        .groupBy("year", "destination")
        .agg(
            F.count("*").alias("num_flights"),
            F.round(F.avg("elapsed_minutes"), 2).alias("avg_minutes"),
            F.min("elapsed_minutes").alias("min_minutes"),
            F.max("elapsed_minutes").alias("max_minutes")
        )
        .orderBy("year", F.desc("num_flights"))
    )

    flight_stats.show(30)

+----+-----------+-----------+-----------+-----------+-----------+
|year|destination|num_flights|avg_minutes|min_minutes|max_minutes|
+----+-----------+-----------+-----------+-----------+-----------+
|2014|        LAX|       5990|     195.25|        152|        331|
|2014|        ORD|       5790|     134.67|        105|        274|
|2014|        SAT|       5147|       63.4|         45|        178|
|2014|        AUS|       4965|      54.73|         40|        165|
|2014|        LGA|       4544|     194.32|        157|        327|
|2014|        LAS|       4034|     172.16|        144|        312|
|2014|        MIA|       3914|      160.7|        135|        303|
|2014|        SFO|       3796|     225.02|        191|        334|
|2014|        MCO|       3655|     146.87|        124|        267|
|2014|        DCA|       3492|     166.02|        133|        282|
|2014|        DEN|       3485|     119.12|         97|        237|
|2014|        ATL|       3344|     121.32|         98|        

In [59]:
if DATA_DIR is not None:
    PROJECT_OUTPUT = str(Path.cwd() / "flight_clean_stats.parquet")

    (
        flight_stats.write
        .mode("overwrite")
        .parquet(PROJECT_OUTPUT)
    )

    print("Đã ghi:", PROJECT_OUTPUT)

Đã ghi: /content/flight_clean_stats.parquet


## Bài tập về nhà

### Bài 1 - Schema

Tạo schema cho dữ liệu:

| product_id | product_name | price | quantity |
|---|---|---:|---:|

Yêu cầu:

- `product_id`: Integer
- `product_name`: String
- `price`: Double
- `quantity`: Integer

---

### Bài 2 - Cleaning

Với `DallasCouncilVoters.csv.gz`:

1. loại `VOTER_NAME` null;
2. trim khoảng trắng;
3. đổi tên thành uppercase;
4. tách tên thành array;
5. đếm số thành phần của tên.

---

### Bài 3 - Conditional Column

Tạo cột `flight_type`:

- `< 120 phút`: `Short`
- `120-240 phút`: `Medium`
- `> 240 phút`: `Long`

---

### Bài 4 - Parquet

1. đọc 4 file chuyến bay;
2. union;
3. ghi Parquet;
4. đọc lại;
5. so sánh schema và số dòng.

---

### Bài 5 - Cache

Thực hiện cùng một phép `groupBy()`:

1. trước khi cache;
2. sau khi cache;
3. `unpersist()` sau khi dùng.

Quan sát thời gian chạy.

---

### Bài 6 - Broadcast Join

Tạo bảng tra cứu mã airport nhỏ và join với dữ liệu chuyến bay.

So sánh:

```python
df.join(lookup_df, ...)
```

và

```python
df.join(F.broadcast(lookup_df), ...)
```

Dùng `explain()` để quan sát kế hoạch.

---

### Bài 7 - Validation

Với Dallas Council Votes:

1. xác định danh sách `VOTE CAST`;
2. tạo danh sách giá trị hợp lệ;
3. dùng join để giữ valid rows;
4. dùng `left_anti` để lấy invalid rows.

### Gợi ý Bài 3

In [60]:
if DATA_DIR is not None:
    exercise_flight = clean_flights.withColumn(
        "flight_type",
        F.when(F.col("elapsed_minutes") < 120, "Short")
         .when(F.col("elapsed_minutes") <= 240, "Medium")
         .otherwise("Long")
    )

    exercise_flight.groupBy("flight_type").count().show()

+-----------+------+
|flight_type| count|
+-----------+------+
|     Medium|382156|
|       Long| 25823|
|      Short|166017|
+-----------+------+



## Tổng kết

Trình tự nội dung của notebook:

```text
Data Cleaning
   ↓
Schema
   ↓
Immutability + Lazy Processing
   ↓
Parquet
   ↓
DataFrame Column Operations
   ↓
when / otherwise
   ↓
UDF
   ↓
Partition + ID
   ↓
Cache
   ↓
Import Performance
   ↓
Execution Plan / Shuffle / Broadcast
   ↓
Data Pipeline
   ↓
Validation
   ↓
Clean Output
```

## Các nguyên tắc quan trọng

1. Định nghĩa schema rõ ràng khi biết trước cấu trúc dữ liệu.
2. Nhớ rằng DataFrame là immutable và Spark thực thi lazy.
3. Ưu tiên built-in Spark functions trước Python UDF.
4. Tránh shuffle không cần thiết.
5. Broadcast bảng nhỏ khi join nếu phù hợp.
6. Chỉ cache khi DataFrame thực sự được tái sử dụng.
7. Dùng Parquet cho dữ liệu trung gian/phân tích khi phù hợp.
8. Data cleaning cần đi kèm validation.
9. Nên lưu riêng dữ liệu invalid để kiểm tra thay vì xóa âm thầm.
10. Dùng `explain()` để hiểu kế hoạch Spark thực thi.

In [61]:
# Khi hoàn thành toàn bộ notebook có thể giải phóng tài nguyên:
spark.stop()